# GBIS 데이터 불러오기 (팀원용)

이 노트북은 개발 환경 설정 없이 GBIS 데이터를 Google Colab에서 DataFrame으로 불러옵니다.

처음 한 번만 다음 작업을 해주세요.

1. Colab 왼쪽의 **열쇠(Secrets)** 아이콘을 누릅니다.
2. 이름이 `GBIS_API_KEY`인 새 보안 비밀을 만들고 전달받은 개인 키를 입력합니다.
3. 이 노트북에서 해당 보안 비밀에 대한 **Notebook access**를 켭니다.
4. 상단 메뉴에서 **런타임 → 모두 실행**을 누릅니다.

최초 실행은 전체 이력을 내려받으므로 시간이 걸릴 수 있습니다. 이후에는 Google Drive의 캐시를 복원하고 새 데이터만 받습니다.

In [5]:
# 아래 값은 특별한 경우가 아니면 그대로 사용하세요.
API_BASE_URL = "https://161.33.212.6"  # @param {type:"string"}
DRIVE_CACHE_PATH = "/content/drive/MyDrive/GBIS/gbis_api_cache.sqlite3"  # @param {type:"string"}
ROUTE_IDS = ""  # @param {type:"string"}
HISTORY_FROM = ""  # @param {type:"string"}

import shutil
import subprocess
import sys
from pathlib import Path

from google.colab import drive, userdata

try:
    API_KEY = userdata.get("GBIS_API_KEY")
except Exception as exc:
    raise RuntimeError(
        "왼쪽 열쇠 아이콘에서 GBIS_API_KEY를 만들고 Notebook access를 켜주세요."
    ) from exc
if not API_KEY:
    raise RuntimeError("Colab Secrets의 GBIS_API_KEY가 비어 있습니다.")

REPO_DIR = Path("/content/gbis_team_repo")
REPO_URL = "https://github.com/khuda-data/10th-toy-team4.git"
if (REPO_DIR / ".git").is_dir():
    subprocess.run(
        ["git", "-C", str(REPO_DIR), "pull", "--ff-only"],
        check=True,
    )
else:
    subprocess.run(
        ["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)],
        check=True,
    )
subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "-r",
        str(REPO_DIR / "requirements-client.txt"),
    ],
    check=True,
)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

drive.mount("/content/drive")
DRIVE_CACHE = Path(DRIVE_CACHE_PATH)
LOCAL_CACHE = Path("/content/gbis_api_cache.sqlite3")
DRIVE_CACHE.parent.mkdir(parents=True, exist_ok=True)
CACHE_RESTORED = DRIVE_CACHE.is_file()
if CACHE_RESTORED:
    shutil.copy2(DRIVE_CACHE, LOCAL_CACHE)
    print("✅ Google Drive에서 기존 캐시를 복원했습니다.")
else:
    print("ℹ️ 첫 실행입니다. 서버의 전체 이력을 내려받습니다.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Google Drive에서 기존 캐시를 복원했습니다.


In [6]:
from IPython.display import display
from gbis_client import GBISApiCache

requested_route_ids = tuple(
    value.strip() for value in ROUTE_IDS.split(",") if value.strip()
)
cache = GBISApiCache(
    base_url=API_BASE_URL,
    api_key=API_KEY,
    cache_path=LOCAL_CACHE,
)
history_counts = {}
try:
    print("1/4 노선 목록을 최신화합니다.")
    cache.refresh_routes()
    routes_df = cache.routes_df()
    available_route_ids = set(routes_df["route_id"].astype(str))
    if requested_route_ids:
        missing_route_ids = set(requested_route_ids) - available_route_ids
        if missing_route_ids:
            raise ValueError(f"서버에 없는 route_id입니다: {sorted(missing_route_ids)}")
        route_ids = requested_route_ids
    else:
        route_ids = tuple(str(value) for value in routes_df["route_id"].tolist())

    print("2/4 정류장 정보를 최신화합니다.")
    station_count_by_route = dict(zip(routes_df["route_id"].astype(str), routes_df["station_count"]))
    station_counts = {
        route_id: cache.refresh_stations(route_id)
        for route_id in route_ids
        if int(station_count_by_route.get(route_id, 0)) > 0
    }

    print("3/4 최신 차량 위치를 갱신합니다.")
    cache.refresh_latest()

    print("4/4 차량 위치 이력을 동기화합니다.")
    for index, route_id in enumerate(route_ids, 1):
        mode = "증분" if CACHE_RESTORED else "최초 전체"
        print(f"  [{index}/{len(route_ids)}] {route_id}: {mode} 동기화 중...")
        history_counts[route_id] = cache.refresh_full_history(route_id)

    routes_df = cache.routes_df()
    stations_df = cache.stations_df()
    latest_df = cache.latest_locations_df()
    history_df = cache.history_df(from_at=HISTORY_FROM or None)
    cache_status_df = cache.cache_status_df()
    if requested_route_ids:
        stations_df = stations_df[stations_df["route_id"].isin(route_ids)].reset_index(drop=True)
        latest_df = latest_df[latest_df["route_id"].isin(route_ids)].reset_index(drop=True)
        history_df = history_df[history_df["route_id"].isin(route_ids)].reset_index(drop=True)
finally:
    cache.close()
    if LOCAL_CACHE.is_file():
        shutil.copy2(LOCAL_CACHE, DRIVE_CACHE)
        print(f"✅ 캐시를 Google Drive에 저장했습니다: {DRIVE_CACHE}")

print("\n동기화 완료")
print(f"- routes_df: {len(routes_df):,}행")
print(f"- stations_df: {len(stations_df):,}행")
print(f"- latest_df: {len(latest_df):,}행")
print(f"- history_df: {len(history_df):,}행")
display(history_df.head())

1/4 노선 목록을 최신화합니다.
2/4 정류장 정보를 최신화합니다.
3/4 최신 차량 위치를 갱신합니다.
4/4 차량 위치 이력을 동기화합니다.
  [1/7] 200000104: 증분 동기화 중...
  [2/7] 218000010: 증분 동기화 중...
  [3/7] 219000013: 증분 동기화 중...
  [4/7] 219000016: 증분 동기화 중...
  [5/7] 222000074: 증분 동기화 중...
  [6/7] 222000075: 증분 동기화 중...
  [7/7] 228000174: 증분 동기화 중...
✅ 캐시를 Google Drive에 저장했습니다: /content/drive/MyDrive/GBIS/gbis_api_cache.sqlite3

동기화 완료
- routes_df: 7행
- stations_df: 615행
- latest_df: 48행
- history_df: 2,634행


,route_id,vehicle_id,observed_at,query_time,plate_no,route_type_code,station_id,station_seq,station_name,remaining_seats,crowded,low_plate,state_code,tagless_code,cached_at_utc
0,219000013,218000165,2026-08-09T21:32:56+09:00,2026-08-09 21:33:10.452,경기73아1134,11,219000561,52,대화역(중),41,1,0,1,1,2026-08-09T14:45:44+00:00
1,219000013,218000166,2026-08-09T21:32:56+09:00,2026-08-09 21:33:10.452,경기73아1135,11,219000383,47,마두역(중),31,1,0,1,1,2026-08-09T14:45:44+00:00
2,219000013,218000176,2026-08-09T21:32:56+09:00,2026-08-09 21:33:10.452,경기73아1152,11,218000320,44,고양경찰서.토당청소년수련관(중),37,1,0,0,1,2026-08-09T14:45:44+00:00
3,219000013,218000179,2026-08-09T21:32:56+09:00,2026-08-09 21:33:10.452,경기73아1155,11,218000319,43,행신초등학교.더자인병원(중),38,1,0,1,1,2026-08-09T14:45:44+00:00
4,219000013,218000180,2026-08-09T21:32:56+09:00,2026-08-09 21:33:10.452,경기73아1156,11,218000078,42,행신동(중),40,1,0,0,1,2026-08-09T14:45:44+00:00


## 사용할 수 있는 DataFrame

- `routes_df`: 노선 목록과 수집 범위
- `stations_df`: 노선별 정류장 순서와 위치
- `latest_df`: 현재 운행 차량별 최신 위치와 잔여좌석
- `history_df`: 전체 또는 지정 시각 이후의 차량 위치 이력
- `cache_status_df`: 데이터별 마지막 갱신 완료 시각

예를 들어 잔여좌석이 0인 기록은 `history_df[history_df["remaining_seats"] == 0]`으로 확인할 수 있습니다.

In [7]:
routes_df.head()

,route_id,station_count,observation_count,first_collected_at,last_collected_at,cached_at_utc
0,200000104,86,374,2026-08-09T21:32:57+09:00,2026-08-09T23:50:03+09:00,2026-08-09T14:53:24+00:00
1,218000010,96,436,2026-08-09T21:32:57+09:00,2026-08-09T23:50:03+09:00,2026-08-09T14:53:24+00:00
2,219000013,55,828,2026-08-09T21:32:56+09:00,2026-08-09T23:50:02+09:00,2026-08-09T14:53:24+00:00
3,219000016,77,503,2026-08-09T21:32:57+09:00,2026-08-09T23:50:02+09:00,2026-08-09T14:53:24+00:00
4,222000074,140,177,2026-08-09T21:32:57+09:00,2026-08-09T23:50:02+09:00,2026-08-09T14:53:24+00:00
